# ONIA — 2.5. Arborii de decizie

Clasificare, regresie, impuritate Gini, implementare de la zero și scikit-learn. Lecție în limba română.

## Calculul Gini de la zero

In [ ]:
def gini(etichete):
    if len(etichete) == 0:
        return 0.0
    frecvente = {}
    for clasa in etichete:
        frecvente[clasa] = frecvente.get(clasa, 0) + 1
    rezultat = 1.0
    for nr in frecvente.values():
        p = nr / len(etichete)
        rezultat -= p * p
    return rezultat

print(gini([0, 0, 0, 0]))  # 0.0
print(gini([0, 0, 1, 1]))  # 0.5
print(gini([0] * 8 + [1] * 2))  # 0.32


## Entropia în Python

In [ ]:
from math import log2

def entropie(etichete):
    n = len(etichete)
    if n == 0:
        return 0.0
    frecvente = {}
    for y in etichete:
        frecvente[y] = frecvente.get(y, 0) + 1
    h = 0.0
    for nr in frecvente.values():
        p = nr / n
        h -= p * log2(p)
    return h

print(round(entropie([0, 0, 0, 1]), 3))  # 0.811


## De la zero: alegerea celui mai bun prag

In [ ]:
def gini_binar(etichete):
    if not etichete:
        return 0.0
    p = sum(etichete) / len(etichete)
    return 2 * p * (1 - p)

def cel_mai_bun_prag(x, y):
    valori = sorted(set(x))
    scor_minim = float("inf")
    rezultat = None
    for i in range(len(valori) - 1):
        prag = (valori[i] + valori[i + 1]) / 2
        st = [y[j] for j in range(len(x)) if x[j] <= prag]
        dr = [y[j] for j in range(len(x)) if x[j] > prag]
        scor = (len(st) * gini_binar(st) +
                len(dr) * gini_binar(dr)) / len(y)
        if scor < scor_minim:
            scor_minim = scor
            rezultat = prag
    return rezultat, scor_minim

x = [1, 2, 3, 7, 8, 9]
y = [0, 0, 0, 1, 1, 1]
print(cel_mai_bun_prag(x, y))  # (5.0, 0.0)


## Implementare didactică a unui arbore CART binar

In [ ]:
def gini_lista(valori):
    if not valori:
        return 0.0
    f = {}
    for y in valori:
        f[y] = f.get(y, 0) + 1
    return 1 - sum((nr / len(valori)) ** 2 for nr in f.values())

def cauta_split(X, y, indici, min_frunza=1):
    cel_mai_bun = None
    scor_minim = gini_lista([y[i] for i in indici])
    for j in range(len(X[0])):
        valori = sorted(set(X[i][j] for i in indici))
        for t in range(len(valori) - 1):
            prag = (valori[t] + valori[t + 1]) / 2
            st = [i for i in indici if X[i][j] <= prag]
            dr = [i for i in indici if X[i][j] > prag]
            if min(len(st), len(dr)) < min_frunza:
                continue
            scor = (len(st) * gini_lista([y[i] for i in st]) +
                    len(dr) * gini_lista([y[i] for i in dr])) / len(indici)
            if scor < scor_minim - 1e-12:
                scor_minim = scor
                cel_mai_bun = (j, prag, st, dr)
    return cel_mai_bun

def construieste_arbore(X, y, indici, adancime=0,
                        max_adancime=3, min_frunza=1):
    frecvente = {}
    for i in indici:
        frecvente[y[i]] = frecvente.get(y[i], 0) + 1
    clasa = max(sorted(frecvente), key=lambda c: frecvente[c])
    nod = {"clasa": clasa, "n": len(indici),
           "exemple": indici[:], "frecvente": frecvente}
    if adancime == max_adancime or len(frecvente) == 1:
        return nod
    split = cauta_split(X, y, indici, min_frunza)
    if split is None:
        return nod
    j, prag, st, dr = split
    nod["caracteristica"] = j
    nod["prag"] = prag
    nod["stanga"] = construieste_arbore(
        X, y, st, adancime + 1, max_adancime, min_frunza)
    nod["dreapta"] = construieste_arbore(
        X, y, dr, adancime + 1, max_adancime, min_frunza)
    return nod

def prezice(nod, x):
    while "stanga" in nod:
        if x[nod["caracteristica"]] <= nod["prag"]:
            nod = nod["stanga"]
        else:
            nod = nod["dreapta"]
    return nod["clasa"]

X = [[1, 2], [2, 4], [3, 8], [5, 3],
     [6, 6], [8, 8], [9, 2], [9, 7]]
y = [0, 0, 1, 0, 1, 1, 0, 1]
arbore = construieste_arbore(X, y, list(range(len(X))))
print(prezice(arbore, [7, 6]))


## Frecvențele relative într-o frunză

In [ ]:
def probabilitati_frunza(nod):
    nr = nod["n"]
    return {c: f / nr for c, f in nod["frecvente"].items()}

frunza = {"n": 10, "frecvente": {"pisica": 7, "caine": 2, "iepure": 1}}
print(probabilitati_frunza(frunza))


## Calculul impurității unui nod de regresie

In [ ]:
def mse_valori(valori):
    medie = sum(valori) / len(valori)
    return sum((v - medie) ** 2 for v in valori) / len(valori)

print(round(mse_valori([10, 12, 20]), 3))  # 18.667
print(sum([10, 12, 20]) / 3)              # 14.0


## Regresie de la zero, un singur atribut

In [ ]:
def mse(y):
    m = sum(y) / len(y)
    return sum((a - m) ** 2 for a in y) / len(y)

def arbore_regresie_1d(x, y, adancime=0, limita=3):
    nod = {"valoare": sum(y) / len(y), "n": len(y)}
    if adancime >= limita or len(set(x)) <= 1:
        return nod
    praguri = sorted(set(x))
    scor_optim = mse(y)
    alegere = None
    for j in range(len(praguri) - 1):
        prag = (praguri[j] + praguri[j + 1]) / 2
        st = [i for i in range(len(x)) if x[i] <= prag]
        dr = [i for i in range(len(x)) if x[i] > prag]
        scor = (len(st) * mse([y[i] for i in st]) +
                len(dr) * mse([y[i] for i in dr])) / len(y)
        if scor < scor_optim - 1e-12:
            scor_optim = scor
            alegere = (prag, st, dr)
    if alegere is not None:
        prag, st, dr = alegere
        nod["prag"] = prag
        nod["stanga"] = arbore_regresie_1d(
            [x[i] for i in st], [y[i] for i in st], adancime+1, limita)
        nod["dreapta"] = arbore_regresie_1d(
            [x[i] for i in dr], [y[i] for i in dr], adancime+1, limita)
    return nod

def prezice_regresie(nod, x):
    while "stanga" in nod:
        nod = nod["stanga"] if x <= nod["prag"] else nod["dreapta"]
    return nod["valoare"]

arb = arbore_regresie_1d([1,2,3,7,8,9], [3,4,5,15,16,17])
print(round(prezice_regresie(arb, 7.5), 2))


## Categorii text și numere într-un Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

X = [["rosu", 2.0], ["verde", 4.0], ["albastru", 8.0],
     ["rosu", 7.0], ["verde", 1.5], ["albastru", 5.5]]
y = [0, 0, 1, 1, 0, 1]
pre = ColumnTransformer([
    ("culori", OneHotEncoder(handle_unknown="ignore"), [0]),
    ("numere", "passthrough", [1]),
])
model = Pipeline([
    ("preprocesare", pre),
    ("arbore", DecisionTreeClassifier(max_depth=3, random_state=42))
])
model.fit(X, y)
print(model.predict([["rosu", 6.0]]))


## Arbore de clasificare pentru Iris

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25,
    random_state=42, stratify=iris.target
)
model = DecisionTreeClassifier(
    criterion="gini", max_depth=3,
    min_samples_leaf=2, random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred,
      target_names=iris.target_names))


## Traseul dintr-un DecisionTreeClassifier

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

iris = load_iris()
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(iris.data, iris.target)
exemplu = iris.data[[0]]
indicii = model.decision_path(exemplu).indices
frunza = model.apply(exemplu)[0]
print("Noduri vizitate:", indicii.tolist())
print("Frunză:", frunza)
for nod in indicii:
    j = model.tree_.feature[nod]
    if j >= 0:
        prag = model.tree_.threshold[nod]
        print(f"Nod {nod}: {iris.feature_names[j]} <= {prag:.3f}?")
print("Predicție:", model.predict(exemplu)[0])


## DecisionTreeRegressor pe valori numerice

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = [[1], [2], [3], [5], [6], [8], [9]]
y_train = [3.1, 4.0, 4.5, 8.2, 9.1, 11.8, 12.0]
model = DecisionTreeRegressor(
    max_depth=2, min_samples_leaf=2, random_state=42
)
model.fit(X_train, y_train)
X_test = [[2.5], [7.0]]
y_test = [4.3, 10.2]
predictii = model.predict(X_test)
print("Predicții:", predictii.round(2).tolist())
print("MAE:", round(mean_absolute_error(y_test, predictii), 3))
print("MSE:", round(mean_squared_error(y_test, predictii), 3))


## Cross-validation pentru hiperparametrii arborelui

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
parametri = {
    "max_depth": [2, 3, 4, 5],
    "min_samples_leaf": [1, 2, 4],
    "ccp_alpha": [0.0, 0.001, 0.01]
}
cautare = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=parametri, cv=4, scoring="accuracy"
)
cautare.fit(X_train, y_train)
print("Parametri aleși:", cautare.best_params_)
print("Accuracy test:", round(cautare.score(X_test, y_test), 3))
